[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mauriciorslrv/DS_basics/blob/main/03%20Aprendizaje%20M%C3%A1quina/notebooks/06_Evaluacion_Modelos_Bank_Marketing.ipynb)

# 06 · Evaluar modelos de clasificación

> **Pregunta operativa:** con capacidad limitada de llamadas, ¿cómo comparamos dos modelos y elegimos un umbral sin mirar el test final?

Continuamos con el dataset real **UCI Bank Marketing**. El caso se formula antes de la llamada, así que excluimos **duration** y **campaign**: duration sólo existe después del contacto y campaign resume los contactos realizados durante la campaña. La evaluación no demuestra que llamar cause una suscripción.

📊 [Ficha UCI y licencia CC BY 4.0](https://archive.ics.uci.edu/dataset/222/bank+marketing) · DOI 10.24432/C5K306.

## 1. Cargar, definir objetivo y preparar el sentinel

In [ ]:
%pip -q install ucimlrepo

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from ucimlrepo import fetch_ucirepo

bank = fetch_ucirepo(id=222)
X = bank.data.features.copy()
y = (bank.data.targets.iloc[:, 0].astype(str).str.lower() == "yes").astype(int)

# La duración de la llamada no está disponible antes de llamar.
# duration se conoce después; campaign resume contactos de toda la campaña.
X = X.drop(columns=["duration", "campaign"], errors="ignore").copy()

# 999 significa que no hubo contacto previo, no que pasaron 999 días.
if "pdays" in X.columns:
    X["not_previously_contacted"] = (X["pdays"] == 999).astype(int)
    X["pdays"] = X["pdays"].replace(999, np.nan)

print("Dimensiones:", X.shape)
print("Tasa positiva:", round(y.mean(), 3))
print("Proporción de clases:")
display(y.value_counts(normalize=True).rename(index={0: "no", 1: "yes"}))

## 2. Reservar test y crear train/validación

El test final se separa antes de comparar modelos o umbrales. Dentro del desarrollo, apartamos validación para decidir qué alternativa llevar al test. Las particiones son estratificadas para conservar la proporción de clases.

In [ ]:
from sklearn.model_selection import train_test_split

X_dev, X_test, y_dev, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
X_train, X_valid, y_train, y_valid = train_test_split(
    X_dev, y_dev, test_size=0.25, random_state=42, stratify=y_dev
)
print("Tamaños train / valid / test:", len(X_train), len(X_valid), len(X_test))
print("Tasas positivas:", *(round(part.mean(), 3) for part in [y_train, y_valid, y_test]))

## 3. Crear pipelines comparables

Usaremos regresión logística como modelo lineal y un árbol de decisión como contraste no lineal. Se aplica la misma limpieza de columnas a ambos. Imputación, escalado y codificación ocurren dentro de cada pipeline.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

categorical = X_train.select_dtypes(include=["object", "category"]).columns.tolist()
numeric = [c for c in X_train.columns if c not in categorical]

preprocess = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scale", StandardScaler())
    ]), numeric),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]), categorical)
])

models = {
    "Regresión logística": Pipeline([
        ("prep", preprocess),
        ("model", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42))
    ]),
    "Árbol de decisión": Pipeline([
        ("prep", preprocess),
        ("model", DecisionTreeClassifier(
            max_depth=8, min_samples_leaf=30, class_weight="balanced", random_state=42
        ))
    ])
}

## 4. Validación cruzada sólo sobre entrenamiento

La validación cruzada compara estabilidad en varios pliegues del conjunto de entrenamiento. ROC-AUC mide ordenamiento de puntuaciones; average precision resume precisión-recall para la clase positiva. Ninguna equivale por sí sola a beneficio de negocio.

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_validate

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
scoring = {"roc_auc": "roc_auc", "avg_precision": "average_precision", "f1": "f1"}
cv_rows = []

for name, estimator in models.items():
    scores = cross_validate(
        estimator, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1
    )
    cv_rows.append({
        "modelo": name,
        **{metric: scores["test_" + metric].mean() for metric in scoring}
    })

pd.DataFrame(cv_rows).set_index("modelo").round(3)

## 5. Elegir el umbral con validación

El umbral 0.50 no es una regla universal. En este ejemplo elegimos modelo y umbral con F1 en el conjunto de validación. En una aplicación real, la métrica debe representar costos, capacidad y consecuencias para las personas.

In [ ]:
from sklearn.metrics import f1_score

thresholds = np.linspace(0.10, 0.90, 17)
validation_choices = {}

for name, estimator in models.items():
    estimator.fit(X_train, y_train)
    probabilities = estimator.predict_proba(X_valid)[:, 1]
    f1_values = [
        f1_score(y_valid, probabilities >= threshold, zero_division=0)
        for threshold in thresholds
    ]
    best = int(np.argmax(f1_values))
    validation_choices[name] = {
        "estimator": estimator,
        "threshold": float(thresholds[best]),
        "f1_valid": float(f1_values[best])
    }
    print(name, "umbral:", round(thresholds[best], 2),
          "F1 validación:", round(f1_values[best], 3))

chosen_name = max(validation_choices, key=lambda name: validation_choices[name]["f1_valid"])
chosen = validation_choices[chosen_name]
print("\nSeleccionado con validación:", chosen_name)

## 6. Evaluación final en test

Ahora aplicamos la alternativa elegida al test, que no participó en la validación cruzada ni en escoger umbral. Reportamos varias métricas porque cada una describe un aspecto distinto.

In [ ]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, balanced_accuracy_score,
    ConfusionMatrixDisplay, classification_report
)

test_probability = chosen["estimator"].predict_proba(X_test)[:, 1]
test_prediction = (test_probability >= chosen["threshold"]).astype(int)

final_metrics = {
    "accuracy": accuracy_score(y_test, test_prediction),
    "balanced accuracy": balanced_accuracy_score(y_test, test_prediction),
    "precision": precision_score(y_test, test_prediction, zero_division=0),
    "recall": recall_score(y_test, test_prediction, zero_division=0),
    "F1": f1_score(y_test, test_prediction, zero_division=0),
    "ROC-AUC": roc_auc_score(y_test, test_probability),
    "average precision": average_precision_score(y_test, test_probability)
}
print("Modelo:", chosen_name, "| Umbral:", round(chosen["threshold"], 2))
pd.Series(final_metrics).round(3)

In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_test, test_prediction, display_labels=["No", "Sí"], cmap="Blues", values_format="d"
)
plt.title(f"Test final · {chosen_name}")
plt.show()

print(classification_report(
    y_test, test_prediction, target_names=["no suscribe", "suscribe"], zero_division=0
))

## 7. Leer el intercambio entre precision y recall

Reducir el umbral suele incluir más casos positivos, aumentando recall pero también los falsos positivos. La curva precisión-recall ayuda a visualizar el intercambio. La línea base de prevalencia sirve de referencia visual; no sustituye una política operativa.

In [ ]:
from sklearn.metrics import PrecisionRecallDisplay

PrecisionRecallDisplay.from_estimator(
    chosen["estimator"], X_test, y_test, name=chosen_name
)
plt.title("Curva precisión-recall · test final")
plt.show()
print("Prevalencia positiva en test:", round(y_test.mean(), 3))

## 🧠 Interpretación y límites

- CV y validación se usaron para comparar y elegir; el test final se usó una vez para estimar generalización en esta muestra.
- La clase positiva es menos frecuente; accuracy no basta.
- El umbral óptimo depende de recursos y costos, que este dataset no especifica.
- La suscripción observada no demuestra que la llamada la haya causado.
- El conjunto es histórico y de una sola institución; antes de usarlo se requiere evaluación con datos actuales, revisión de privacidad y análisis de impactos.

## 🧪 Mini reto

Define una capacidad concreta: por ejemplo, contactar como máximo al 10 % con mayor puntuación. Evalúa precision entre ese grupo en validación y test. Compara con la prevalencia base y explica qué experimento permitiría medir el efecto de la política de contacto.

## 📚 Fuentes y atribución

- Moro, S., Cortez, P. & Rita, P. (2014). UCI Bank Marketing. [Ficha y licencia CC BY 4.0](https://archive.ics.uci.edu/dataset/222/bank+marketing), DOI 10.24432/C5K306.
- Scikit-learn: [validación cruzada](https://scikit-learn.org/stable/modules/cross_validation.html), [errores comunes y data leakage](https://scikit-learn.org/stable/common_pitfalls.html), [métricas de clasificación](https://scikit-learn.org/stable/modules/model_evaluation.html#classification-metrics).